### Project 5 - Solutions

For this project we need to read CSV files lazily and produce each data row as a named tuple.

We will solve the project twice:

1. with a class that implements the context manager protocol;
2. with a generator function decorated by `contextlib.contextmanager`.

### Project 5 - Goal 1

First, let's inspect the first two lines of each project file.

This lets us confirm that:

- the first row contains field names;
- the files do not necessarily use the same delimiter.

In [1]:
from pathlib import Path

f_names = "cars.csv", "personal_info.csv"

for f_name in f_names:
    path = Path(f_name)

    if not path.exists():
        print("{} was not found.".format(f_name))
        print("-----------------")
        continue

    with path.open("r", encoding="utf-8-sig", newline="") as f:
        print(next(f), end="")
        print(next(f), end="")

    print("\n-----------------")

Car;MPG;Cylinders;Displacement;Horsepower;Weight;Acceleration;Model;Origin
Chevrolet Chevelle Malibu;18.0;8;307.0;130.0;3504.;12.0;70;US

-----------------
ssn,first_name,last_name,gender,language
100-53-9824,Sebastiano,Tester,Male,Icelandic

-----------------


The two files use different delimiters:

- `cars.csv` uses a semicolon;
- `personal_info.csv` uses a comma.

Since our context manager should only require a file name, we should not hardcode either delimiter.

The `csv.Sniffer` class can examine a sample and make a best guess about the CSV dialect.

Let's import the tools we will need:

In [2]:
import csv
from collections import namedtuple
from contextlib import contextmanager
from itertools import islice

We will first create a small helper that detects a usable dialect.

There are two details worth handling:

1. after reading a sample, we must rewind the file with `seek(0)`;
2. `Sniffer` is heuristic, so we use a conservative fallback when it cannot determine a valid dialect.

In [3]:
COMMON_DELIMITERS = ",;\t|"
SAMPLE_SIZE = 8 * 1024


def get_dialect(file_obj):
    """Detect the CSV dialect and leave the file positioned at the start."""
    sample = file_obj.read(SAMPLE_SIZE)
    file_obj.seek(0)

    if not sample:
        raise ValueError("The CSV file is empty.")

    try:
        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=COMMON_DELIMITERS,
        )
    except csv.Error:
        dialect = csv.excel

    delimiter = getattr(dialect, "delimiter", None)

    if not isinstance(delimiter, str) or len(delimiter) != 1:
        dialect = csv.excel

    try:
        csv.reader([], dialect=dialect)
    except (csv.Error, TypeError, ValueError):
        dialect = csv.excel

    return dialect

Now we can use the helper to check the delimiter that was detected for each file:

In [4]:
for f_name in f_names:
    path = Path(f_name)

    if not path.exists():
        continue

    with path.open("r", encoding="utf-8-sig", newline="") as f:
        dialect = get_dialect(f)

    print("{} -> {!r}".format(f_name, dialect.delimiter))

cars.csv -> ';'
personal_info.csv -> ','


We want to create a context manager that, given only a file name:

1. opens the file;
2. detects the CSV dialect;
3. creates a `csv.reader`;
4. reads the header row;
5. creates a named-tuple class from those field names;
6. returns a lazy iterator over the remaining rows;
7. closes the file when the `with` block ends.

We will make one class implement both protocols:

- the **context manager protocol**: `__enter__` and `__exit__`;
- the **iterator protocol**: `__iter__` and `__next__`.

We will preserve the header names exactly as they appear in the CSV file, since the project asks for field names corresponding to the header row.

In [5]:
class FileParser:
    def __init__(self, f_name):
        self.f_name = Path(f_name)
        self._f = None
        self._reader = None
        self._nt = None
        self._headers = None

    def __enter__(self):
        self._f = self.f_name.open(
            "r",
            encoding="utf-8-sig",
            newline="",
        )

        try:
            dialect = get_dialect(self._f)
            self._reader = csv.reader(self._f, dialect=dialect)

            try:
                self._headers = tuple(next(self._reader))
            except StopIteration:
                raise ValueError("The CSV file does not contain a header row.")

            self._nt = namedtuple("Data", self._headers)
            return self

        except Exception:
            self._f.close()
            self._f = None
            raise

    def __exit__(self, exc_type, exc_value, exc_tb):
        if self._f is not None:
            self._f.close()

        self._reader = None
        self._nt = None
        self._headers = None
        return False

    def __iter__(self):
        return self

    def __next__(self):
        if self._reader is None:
            raise StopIteration

        row = next(self._reader)

        if len(row) != len(self._headers):
            raise ValueError(
                "Expected {} fields on CSV line {}, but received {}.".format(
                    len(self._headers),
                    self._reader.line_num,
                    len(row),
                )
            )

        return self._nt(*row)

    @property
    def field_names(self):
        if self._headers is None:
            raise RuntimeError(
                "field_names is available only inside the with block."
            )
        return self._headers

    @property
    def is_closed(self):
        return self._f is None or self._f.closed

Let's use the class with `cars.csv`.

`islice` requests only the first ten records, so the entire file is not loaded into memory.

In [6]:
cars_path = Path("cars.csv")

if cars_path.exists():
    with FileParser(cars_path) as data:
        print("Fields:", data.field_names)
        print()

        for row in islice(data, 10):
            print(row)
else:
    print("cars.csv was not found.")

Fields: ('Car', 'MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model', 'Origin')

Data(Car='Chevrolet Chevelle Malibu', MPG='18.0', Cylinders='8', Displacement='307.0', Horsepower='130.0', Weight='3504.', Acceleration='12.0', Model='70', Origin='US')
Data(Car='Buick Skylark 320', MPG='15.0', Cylinders='8', Displacement='350.0', Horsepower='165.0', Weight='3693.', Acceleration='11.5', Model='70', Origin='US')
Data(Car='Plymouth Satellite', MPG='18.0', Cylinders='8', Displacement='318.0', Horsepower='150.0', Weight='3436.', Acceleration='11.0', Model='70', Origin='US')
Data(Car='AMC Rebel SST', MPG='16.0', Cylinders='8', Displacement='304.0', Horsepower='150.0', Weight='3433.', Acceleration='12.0', Model='70', Origin='US')
Data(Car='Ford Torino', MPG='17.0', Cylinders='8', Displacement='302.0', Horsepower='140.0', Weight='3449.', Acceleration='10.5', Model='70', Origin='US')
Data(Car='Ford Galaxie 500', MPG='15.0', Cylinders='8', Displacement='429.0', Horsep

The same context manager should work with `personal_info.csv`, even though that file uses a different delimiter:

In [7]:
personal_info_path = Path("personal_info.csv")

if personal_info_path.exists():
    with FileParser(personal_info_path) as data:
        print("Fields:", data.field_names)
        print()

        for row in islice(data, 10):
            print(row)
else:
    print("personal_info.csv was not found.")

Fields: ('ssn', 'first_name', 'last_name', 'gender', 'language')

Data(ssn='100-53-9824', first_name='Sebastiano', last_name='Tester', gender='Male', language='Icelandic')
Data(ssn='101-71-4702', first_name='Cayla', last_name='MacDonagh', gender='Female', language='Lao')
Data(ssn='101-84-0356', first_name='Nomi', last_name='Lipprose', gender='Female', language='Yiddish')
Data(ssn='104-22-0928', first_name='Justinian', last_name='Kunzelmann', gender='Male', language='Dhivehi')
Data(ssn='104-84-7144', first_name='Claudianus', last_name='Brixey', gender='Male', language='Afrikaans')
Data(ssn='105-27-5541', first_name='Federico', last_name='Aggett', gender='Male', language='Chinese')
Data(ssn='105-85-7486', first_name='Angelina', last_name='McAvey', gender='Female', language='Punjabi')
Data(ssn='105-91-5022', first_name='Moselle', last_name='Apfel', gender='Female', language='Latvian')
Data(ssn='105-91-7777', first_name='Audi', last_name='Roach', gender='Female', language='Estonian')
Data(

We can also verify that the context manager closes the file automatically:

In [8]:
if cars_path.exists():
    parser = FileParser(cars_path)

    with parser as data:
        print("Inside the context:", data.is_closed)

    print("After the context:", parser.is_closed)
else:
    print("Resource-closing example skipped.")

Inside the context: False
After the context: True


### Project 5 - Goal 2

For Goal 2 we need to reproduce the same behavior using:

- a generator function;
- the `contextmanager` decorator from `contextlib`.

The decorated function will still manage the lifetime of the file.

Inside it, a nested generator will lazily read and convert one row at a time.

In [9]:
@contextmanager
def file_parser(f_name):
    path = Path(f_name)
    f = path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    )

    try:
        dialect = get_dialect(f)
        reader = csv.reader(f, dialect=dialect)

        try:
            headers = tuple(next(reader))
        except StopIteration:
            raise ValueError("The CSV file does not contain a header row.")

        nt = namedtuple("Data", headers)

        def row_iterator():
            for row in reader:
                if len(row) != len(headers):
                    raise ValueError(
                        "Expected {} fields on CSV line {}, but received {}.".format(
                            len(headers),
                            reader.line_num,
                            len(row),
                        )
                    )

                yield nt(*row)

        yield row_iterator()

    finally:
        f.close()

Let's test the generator-based context manager with `cars.csv`:

In [10]:
if cars_path.exists():
    with file_parser(cars_path) as data:
        for row in islice(data, 10):
            print(row)
else:
    print("cars.csv was not found.")

Data(Car='Chevrolet Chevelle Malibu', MPG='18.0', Cylinders='8', Displacement='307.0', Horsepower='130.0', Weight='3504.', Acceleration='12.0', Model='70', Origin='US')
Data(Car='Buick Skylark 320', MPG='15.0', Cylinders='8', Displacement='350.0', Horsepower='165.0', Weight='3693.', Acceleration='11.5', Model='70', Origin='US')
Data(Car='Plymouth Satellite', MPG='18.0', Cylinders='8', Displacement='318.0', Horsepower='150.0', Weight='3436.', Acceleration='11.0', Model='70', Origin='US')
Data(Car='AMC Rebel SST', MPG='16.0', Cylinders='8', Displacement='304.0', Horsepower='150.0', Weight='3433.', Acceleration='12.0', Model='70', Origin='US')
Data(Car='Ford Torino', MPG='17.0', Cylinders='8', Displacement='302.0', Horsepower='140.0', Weight='3449.', Acceleration='10.5', Model='70', Origin='US')
Data(Car='Ford Galaxie 500', MPG='15.0', Cylinders='8', Displacement='429.0', Horsepower='198.0', Weight='4341.', Acceleration='10.0', Model='70', Origin='US')
Data(Car='Chevrolet Impala', MPG='14

And of course it should work equally well with `personal_info.csv`:

In [11]:
if personal_info_path.exists():
    with file_parser(personal_info_path) as data:
        for row in islice(data, 10):
            print(row)
else:
    print("personal_info.csv was not found.")

Data(ssn='100-53-9824', first_name='Sebastiano', last_name='Tester', gender='Male', language='Icelandic')
Data(ssn='101-71-4702', first_name='Cayla', last_name='MacDonagh', gender='Female', language='Lao')
Data(ssn='101-84-0356', first_name='Nomi', last_name='Lipprose', gender='Female', language='Yiddish')
Data(ssn='104-22-0928', first_name='Justinian', last_name='Kunzelmann', gender='Male', language='Dhivehi')
Data(ssn='104-84-7144', first_name='Claudianus', last_name='Brixey', gender='Male', language='Afrikaans')
Data(ssn='105-27-5541', first_name='Federico', last_name='Aggett', gender='Male', language='Chinese')
Data(ssn='105-85-7486', first_name='Angelina', last_name='McAvey', gender='Female', language='Punjabi')
Data(ssn='105-91-5022', first_name='Moselle', last_name='Apfel', gender='Female', language='Latvian')
Data(ssn='105-91-7777', first_name='Audi', last_name='Roach', gender='Female', language='Estonian')
Data(ssn='106-35-1938', first_name='Mackenzie', last_name='Nussey', gen

### A small self-contained test

The project files may not always be present when this notebook is opened elsewhere.

The following test creates temporary CSV files and verifies that both solutions:

- detect comma and semicolon delimiters;
- preserve header names;
- return strings;
- iterate lazily;
- reject malformed rows.

In [12]:
from tempfile import TemporaryDirectory


def write_text(path, text):
    path.write_text(text, encoding="utf-8")


with TemporaryDirectory() as temp_dir_name:
    temp_dir = Path(temp_dir_name)

    cars_test = temp_dir / "cars_test.csv"
    people_test = temp_dir / "people_test.csv"
    malformed_test = temp_dir / "malformed_test.csv"

    write_text(
        cars_test,
        "Car;MPG;Origin\n"
        "Chevrolet Chevelle Malibu;18.0;US\n"
        "Toyota Corolla;31.0;Japan\n",
    )

    write_text(
        people_test,
        "ssn,first_name,last_name\n"
        "100-53-9824,Sebastiano,Tester\n"
        "101-71-4702,Cayla,MacDonagh\n",
    )

    write_text(
        malformed_test,
        "a,b,c\n"
        "1,2\n",
    )

    with FileParser(cars_test) as data:
        first_car = next(data)

        assert data.field_names == ("Car", "MPG", "Origin")
        assert first_car.Car == "Chevrolet Chevelle Malibu"
        assert first_car.MPG == "18.0"
        assert first_car.Origin == "US"
        assert all(isinstance(value, str) for value in first_car)

    with file_parser(people_test) as data:
        first_person = next(data)

        assert first_person.ssn == "100-53-9824"
        assert first_person.first_name == "Sebastiano"
        assert first_person.last_name == "Tester"
        assert all(isinstance(value, str) for value in first_person)

    try:
        with FileParser(malformed_test) as data:
            next(data)
    except ValueError as exc:
        assert "Expected 3 fields" in str(exc)
    else:
        raise AssertionError("The malformed row should have raised ValueError.")

print("All tests passed.")

All tests passed.


### Final comparison

Both solutions satisfy the project requirements.

The class-based version:

```python
with FileParser("cars.csv") as data:
    first_row = next(data)
```

The generator-based version:

```python
with file_parser("cars.csv") as data:
    first_row = next(data)
```

In both cases:

- only the file name is required;
- the delimiter is detected automatically;
- the header creates the named-tuple fields;
- every value remains a string;
- records are produced lazily;
- the file is closed when the context exits.